##### from binary to json strings and load to bronze, we get binary because any data stream in kafka is converted into binary so if you want to use it again you need convert it to string and store it.

In [0]:
from pyspark.sql.functions import * ## importing all functions

In [0]:
#scope name  = hospitalanalyticsvaultscope

In [0]:
# ─────────────────────────────────────────
# SECTION 1: MY AZURE EVENT HUB DETAILS
# ─────────────────────────────────────────

event_hub_namespace = <<"Storageaccount_name">>
# My Event Hubs Host Name

event_hub_name = "<<Event_Hub_Name>>"
# The specific hub name e.g. "patient-flow-eventhub"

event_hub_conn_str = <<"Connection_String">>
# Your secret key — copied from Azure Portal (Shared Access Policies)
#syntax to refer to the key vault

# SECTION 2: KAFKA CONNECTION SETTINGS

kafka_options = {
# kafka option is a dictionary that was created in python and need to be pass into the options
    'kafka.bootstrap.servers': f"{event_hub_namespace}:9093",
    # WHERE to connect my Event Hubs address + port 9093
    # Port 9093 is the standard Kafka SSL port Azure uses

    'subscribe': event_hub_name,
    # WHAT to listen to my specific Event Hub (like tuning to a radio channel)

    'kafka.security.protocol': 'SASL_SSL',
    # HOW to connect securely encrypted connection (like HTTPS for Kafka)

    'kafka.sasl.mechanism': 'PLAIN',
    # Authentication method — sends username/password to verify identity

    'kafka.sasl.jaas.config': f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{event_hub_conn_str}";',
    # THE ACTUAL LOGIN — 
    # username is always "$ConnectionString" (Azure standard)
    # password is my connection string (my secret key)

    'startingOffsets': 'latest',
    # WHERE to start reading 
    # 'latest' means only read NEW data coming in
    # (not old data already in Event Hubs)

    'failOnDataLoss': 'false'
    # IF data is missing — don't crash the pipeline
    # just keep running (important for stability)
}

#Read from eventhub
raw_df = (spark.readStream
          .format("kafka")
          .options(**kafka_options)  ## converting it from dictionary to options format
          .load()
          )

In [0]:
#Casting data to json
json_df = raw_df.selectExpr("CAST(value AS STRING) as raw_json")  ## converting binary to raw json string

In [0]:
#ADLS configuration (access to the storage account)
spark.conf.set(
  "fs.azure.account.key.<<Storageaccount_name">>.dfs.core.windows.net",
  <<Storage_Access_Key>>
)

#specifying the part to write the data
bronze_path = "abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path"

##patient_flow is a new container I just created that will be a folder inside bronze container

In [0]:
checkpoint_path = "abfss://bronze@hospitalstoragedon.dfs.core.windows.net/_checkpoints/patient_flow"

# Writing stream data to bronze using delta format
(
    json_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)  # ✅ Now points to ADLS
    .start(bronze_path)
)

In [0]:
# # Full secure path to Azure Data Lake Storage (ADLS)
# # where Databricks saves the streaming progress (checkpoint)
# # Format: abfss://container@storageaccount.dfs.core.windows.net/folder/subfolder

# checkpoint_path = "abfss://bronze@hospitalstoragedon.dfs.core.windows.net/_checkpoints/patient_flow"
# #                          ↑            ↑                                      ↑              ↑
# #                      container   storage account                        checkpoint      stream
# #                        name          name                                 folder         name

In [0]:
## To see the records that is inserted to bronze table
display(spark.read.format("delta").load(bronze_path))

raw_json
"{""patient_id"": ""497b056a-c0e1-49bd-bdeb-29532d50d3e9"", ""gender"": ""Male"", ""age"": 49, ""department"": ""Cardiology"", ""admission_time"": ""2026-04-19T14:00:37.941038"", ""discharge_time"": ""2026-04-22T06:00:37.941038"", ""bed_id"": 345, ""hospital_id"": 5}"
"{""patient_id"": ""9013e599-aa05-40f0-b209-923e4f8ef2e9"", ""gender"": ""Male"", ""age"": 53, ""department"": ""Maternity"", ""admission_time"": ""2026-04-19T08:00:38.947885"", ""discharge_time"": ""2026-04-19T19:00:38.947885"", ""bed_id"": 291, ""hospital_id"": 5}"
"{""patient_id"": ""c33365a1-49ff-4bee-bc3b-695c879ec891"", ""gender"": ""Male"", ""age"": 150, ""department"": ""Cardiology"", ""admission_time"": ""2026-04-17T21:00:39.949816"", ""discharge_time"": ""2026-04-18T00:00:39.949816"", ""bed_id"": 230, ""hospital_id"": 2}"
"{""patient_id"": ""abe0110b-bb93-401f-b8a8-40dce360312e"", ""gender"": ""Female"", ""age"": 14, ""department"": ""Surgery"", ""admission_time"": ""2026-04-19T08:00:40.951727"", ""discharge_time"": ""2026-04-21T08:00:40.951727"", ""bed_id"": 61, ""hospital_id"": 1}"
"{""patient_id"": ""bc53ada9-3511-4362-a37c-a4273edc434a"", ""gender"": ""Male"", ""age"": 95, ""department"": ""Oncology"", ""admission_time"": ""2026-04-18T08:00:41.953577"", ""discharge_time"": ""2026-04-20T23:00:41.953577"", ""bed_id"": 134, ""hospital_id"": 2}"
"{""patient_id"": ""a9c5946a-c38c-42a6-8ee2-c83ce27b17de"", ""gender"": ""Male"", ""age"": 119, ""department"": ""Emergency"", ""admission_time"": ""2026-04-17T14:00:42.957535"", ""discharge_time"": ""2026-04-19T21:00:42.957535"", ""bed_id"": 193, ""hospital_id"": 2}"
"{""patient_id"": ""0953e580-e47b-4083-b4f8-9b254c9fcf38"", ""gender"": ""Female"", ""age"": 17, ""department"": ""ICU"", ""admission_time"": ""2026-04-19T08:00:43.959919"", ""discharge_time"": ""2026-04-19T10:00:43.959919"", ""bed_id"": 475, ""hospital_id"": 3}"
"{""patient_id"": ""3fd87634-204b-4306-9004-cbc4e2c4d353"", ""gender"": ""Male"", ""age"": 37, ""department"": ""Maternity"", ""admission_time"": ""2026-04-18T11:00:44.970309"", ""discharge_time"": ""2026-04-18T15:00:44.970309"", ""bed_id"": 303, ""hospital_id"": 1}"
"{""patient_id"": ""8d774b05-17f0-4e61-8e67-31cdedf93d4e"", ""gender"": ""Male"", ""age"": 72, ""department"": ""Surgery"", ""admission_time"": ""2026-04-18T05:00:45.975370"", ""discharge_time"": ""2026-04-19T15:00:45.975370"", ""bed_id"": 308, ""hospital_id"": 4}"
"{""patient_id"": ""87f41406-c7a3-4553-a4fc-15f86841e631"", ""gender"": ""Female"", ""age"": 12, ""department"": ""Surgery"", ""admission_time"": ""2026-04-17T19:00:46.981967"", ""discharge_time"": ""2026-04-19T16:00:46.981967"", ""bed_id"": 266, ""hospital_id"": 6}"
